In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.shape

(50000, 2)

In [6]:
df.isna().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df.shape

(49582, 2)

### Pre-processing


In [9]:
# Converting to Lowercase

df["review"] = df["review"].str.lower()

In [10]:
# Removing the URLs

import re

def remove_urls(text):
    text = re.sub(r"http\S+", "", text) # (pattern, replacement, string) ex - https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

In [9]:
# Removing Punctuations

def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # A-Z, a-z, 0-9  \s
    return text

df["review"] = df["review"].apply(remove_punctuation)

In [10]:
# Removing HTML tags

def remove_html_tags(text):
    text = re.sub(r"<.*?>", "", text) # HTML Tags
    return text

df["review"] = df["review"].apply(remove_html_tags)


In [11]:
# removing the stopwords

#import nltk

#nltk.download("punkt")
#nltk.download("punkt_tab")
#nltk.download("stopwords")



In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [12]:
#example

sample_text = "I like coding in Py"
tokens = word_tokenize(sample_text)
tokens


['I', 'like', 'coding', 'in', 'Py']

In [15]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    
    for word in tokens:
        if word in stop_words:
            text.replace(word, "")
            
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [16]:
df["review"] = df["review"].apply(remove_stopwords)

In [17]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [18]:
# stemming 
# played -> play
# running -> run


# PorterStemming

from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [19]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the film techniqu...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love in the time of money is a v...,positive


In [13]:
# Encoding

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"]) # Positive => 1, negative => 0

In [14]:
y = df["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [15]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. <br /><br />the...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1


In [16]:
# vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

In [17]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5817973 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3120)	0.019429173719885708
  (0, 3098)	0.09151500895941889
  (0, 4442)	0.20052665014215915
  (0, 3147)	0.05755396744514131
  (0, 3696)	0.07069884475520251
  (0, 2066)	0.02383069103421379
  (0, 2823)	0.06076831486537707
  (0, 4440)	0.06018544664467151
  (0, 149)	0.030812630098585387
  (0, 4829)	0.07167608450679853
  (0, 2450)	0.04638547440162319
  (0, 3172)	0.49498849535885786
  (0, 1525)	0.10637449993427063
  (0, 4990)	0.0597025679365566
  (0, 2642)	0.12460207969414304
  (0, 425)	0.03908218519282525
  (0, 2176)	0.07722803723546284
  (0, 4457)	0.022828478254064064
  (0, 292)	0.039576766451677896
  (0, 3712)	0.07963649733711928
  (0, 319)	0.07190332251076077
  (0, 4468)	0.040943870014681326
  (0, 2374)	0.1242214022204563
  (0, 1568)	0.05362151164203855
  (0, 4870)	0.04779984588910038
  :	:
  (49581, 2205)	0.08362037191052366
  (49581, 4608)	0.09415811268743

In [46]:
# Dataset, Dataloader

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [47]:
print("X_train", X_train.shape)
print("X_test", X_test.shape)

X_train (39665, 5000)
X_test (9917, 5000)


In [48]:
import torch
from torch.utils.data import TensorDataset, DataLoader


In [51]:
train_set = TensorDataset(
torch. from_numpy(X_train). float(),
torch. from_numpy(y_train. values) . float()
)

test_set = TensorDataset(
torch. from_numpy(X_test).float(),
torch. from_numpy(y_test.values) . float()
)

TypeError: expected np.ndarray (got csr_matrix)